In [ ]:
import pandas as pd
import torch
from torch import nn
import cv2
from torchvision.io import read_image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
import glob
import time, datetime
from tqdm import tqdm
from torch import optim
from torchsummary import summary
import numpy as np
import segmentation_models_pytorch as smp
from sklearn.metrics import mean_absolute_error
import timm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, recall_score, f1_score

# Prepare data

In [ ]:
csv_train = pd.read_csv('/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Train/Train.csv')
csv_train.head()

In [ ]:
csv_val = pd.read_csv('/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Validation/Validation.csv')
csv_val.head()

In [ ]:
csv_test = pd.read_csv('/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Test/Test.csv')
csv_test.head()

In [ ]:
class CrossDataset(Dataset):
    def __init__(self, train_folder, csv_data, transform=None):
        self.train_folder = train_folder
        self.df = pd.read_csv(csv_data)
        self.transform = transform
        self.data = []
        for i in range(self.df.shape[0]):
            self.data.append(list(self.df.loc[i]))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_file = self.data[idx][0]  
        img_file_path = os.path.join(self.train_folder, img_file)
        img = self.transform(cv2.cvtColor(cv2.imread(img_file_path), cv2.COLOR_BGR2RGB))
        x1 = np.float32(self.data[idx][1])
        x2 = np.float32(self.data[idx][2])
        x3 = np.float32(self.data[idx][3])
        
        output = np.array((x1, x2, x3))
        return img, output, img_file  

In [ ]:
train_folder = "/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Train"
train_csv = '/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Train/Train.csv'

train_transform = transforms.Compose([
    transforms.ToTensor(),  # Convert image to PyTorch tensor
    transforms.Resize((512,512))  # Resize image to 512x512
])

train_data = CrossDataset(train_folder, train_csv, train_transform)
batch_size = 2
train_data_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [ ]:
val_folder = "/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Validation"
val_csv = '/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Validation/Validation.csv'

val_transform = transforms.Compose([
    transforms.ToTensor(),  # Convert image to PyTorch tensor
    transforms.Resize((512,512))  # Resize image to 512x512
])

val_data = CrossDataset(val_folder, val_csv, val_transform)
val_batch_size = 2
val_data_loader = DataLoader(val_data, batch_size=val_batch_size, shuffle=True)

In [ ]:
img, output, filenames = next(iter(train_data_loader))
plt.imshow(np.transpose(img[0], (1, 2, 0)))
print(output[0,:])

In [ ]:
img, output, filenames = next(iter(val_data_loader))
plt.imshow(np.transpose(img[0], (1, 2, 0)))
print(output[0,:])

# Create model

In [ ]:
class HRNetCustomModel(nn.Module):
    def __init__(self):
        super(HRNetCustomModel, self).__init__()
        
        # HRNet Backbone (ใช้จาก timm)
        self.hrnet = timm.create_model('hrnet_w48', pretrained=True, features_only=True)
        
        # Additional Fully Connected Layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1024 * 16 * 16, 1024)  # ปรับขนาดให้ตรงกับ output feature map
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, 3)

    def forward(self, x):
        features = self.hrnet(x)
        x = features[-1]  # ใช้ feature map สุดท้าย
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        return self.fc6(x)

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HRNetCustomModel().to(device)

# Print model summary
from torchinfo import summary
summary(model, input_size=(2, 3, 512, 512))  # Batch size = 2


# Train

In [ ]:
# Hyperparameter
learning_rate = 0.0001
batch_size = 2
num_epochs = 200

# Loss and optimizer
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
img, output, filenames = next(iter(train_data_loader))
img = img.to(device)
pred = model(img)
print(output)
print(pred)

In [ ]:
img, output, filenames = next(iter(val_data_loader))
img = img.to(device)
pred = model(img)
print(output)
print(pred)

In [ ]:

# Paths and hyperparameters
MODEL_SAVE_PATH = 'best_Detection.pth'

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    running_loss = 0.0

    for img, output in tqdm(train_loader, desc="Training"):
        # Move data to the device
        img, output = img.to(device), output.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        pred = model(img)
        loss = criterion(output, pred)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item() * img.size(0)

    # Compute epoch loss
    epoch_loss = running_loss / len(train_loader.dataset)
    return epoch_loss

def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    all_preds, all_outputs = [], []

    with torch.no_grad():  # No gradient calculation
        for img, output in tqdm(val_loader, desc="Validation"):
            # Move data to the device
            img, output = img.to(device), output.to(device)

            # Forward pass
            pred = model(img)
            loss = criterion(output, pred)

            # Accumulate validation loss
            val_loss += loss.item() * img.size(0)

            # Store predictions and outputs
            all_preds.append(pred.cpu().numpy())
            all_outputs.append(output.cpu().numpy())

    # Compute epoch validation loss and MAE
    val_loss /= len(val_loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_outputs = np.concatenate(all_outputs)
    val_mae = mean_absolute_error(all_outputs, all_preds)

    return val_loss, val_mae

def main_training_loop(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    # Initialize tracking variables
    history = []
    best_val_mae = float('inf')
    best_epoch = 0  # Track the epoch of the best model
    start = datetime.datetime.now()

    print(f"Training started at {start}\n")

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")

        # Training phase
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        print(f"Training Loss: {train_loss:.4f}")

        # Validation phase
        val_loss, val_mae = validate_one_epoch(model, val_loader, criterion, device)
        print(f"Validation Loss: {val_loss:.4f}, Validation MAE: {val_mae:.4f}")

        # Update history
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_mae': val_mae})

        # Save the best model and track the best epoch
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch + 1
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"Best model saved at epoch {best_epoch} with MAE {best_val_mae:.4f}")

    # Log total training time
    end = datetime.datetime.now()
    print(f"\nTraining completed at {end}")
    print(f"Total training time: {end - start}")
    print(f"Best Validation MAE: {best_val_mae:.4f} achieved at epoch {best_epoch}")

    return history

# Assuming you have already initialized the following:
# model, train_data_loader, val_data_loader, criterion, optimizer, num_epochs, and device

# Run the training loop
history = main_training_loop(model, train_data_loader, val_data_loader, criterion, optimizer, num_epochs, device) 

In [ ]:
def plot_training_history(history):
    # ดึงข้อมูลจาก history
    epochs = [h['epoch'] for h in history]
    train_losses = [h['train_loss'] for h in history]
    val_losses = [h['val_loss'] for h in history]
    val_maes = [h['val_mae'] for h in history]

    # หาค่า MAE ที่ดีที่สุดและ epoch ที่เกี่ยวข้อง
    best_mae = min(val_maes)
    best_epoch = epochs[val_maes.index(best_mae)]

    # วาดกราฟ Training และ Validation Loss
    plt.figure(figsize=(12, 6))
    
    # กราฟ Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Training Loss', marker='o')
    plt.plot(epochs, val_losses, label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # กราฟ MAE
    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_maes, label='Validation MAE', color='green', marker='o')
    plt.scatter(best_epoch, best_mae, color='red', label=f'Best MAE: {best_mae:.4f} (Epoch {best_epoch})')
    plt.title('Validation Mean Absolute Error (MAE)')
    plt.xlabel('Epochs')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Call the function to plot the history
plot_training_history(history)

# Test

In [ ]:
# Load the model and prepare for evaluation
model.load_state_dict(torch.load('best_HR+reg3(0.870).pth', weights_only=True))
model = model.to(device)
model.eval()

# Define the Test Dataset and DataLoader
test_folder = "/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Test"
test_csv = '/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Rotation images/Test/Test.csv'
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((512, 512))
])

test_data = CrossDataset(test_folder, test_csv, test_transform)
batch_size = 2
test_data_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Define MAE calculation function
def MAE(real, pred):
    real = real.detach().cpu().numpy()
    pred = pred.detach().cpu().numpy()
    mae = np.mean(np.abs(real - pred))
    return mae

# Process all test data and calculate MAE
all_real = []
all_pred = []

for images, labels, filenames in test_data_loader:
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():  # Disable gradient computation for evaluation
        outputs = model(images)
    
    # Collect predictions and ground truths
    all_real.append(labels)
    all_pred.append(outputs)

# Concatenate all predictions and ground truths
all_real = torch.cat(all_real, dim=0)
all_pred = torch.cat(all_pred, dim=0)

# Calculate MAE for all test data
mae_value = MAE(all_real, all_pred)

print(f'MAE (Mean Absolute Error) for the entire test set: {mae_value}')

In [ ]:
img, output, filenames = next(iter(test_data_loader))
img = img.to(device)
pred = model(img)
print(f'GT: {output}')
print(f'Pred: {pred}')

In [ ]:
def cal_distance(point):
    x1, x2, x3 = point
    distance_left = round(x2 - x3)
    distance_right = round(x3 - x1)
    return distance_left, distance_right

def cal_alpha(distance_left, distance_right):
    alpha = (distance_right - distance_left) / (distance_right + distance_left)
    return round(alpha, 3)  


In [ ]:
# คำนวณระยะทางและ alpha
distance_left, distance_right = cal_distance(output[0, :].detach().cpu().numpy())  # ย้าย Tensor ไป CPU ก่อน
print(f'Distance Left: {distance_left}')
print(f'Distance Right: {distance_right}')

alpha = cal_alpha(distance_left, distance_right)  # คำนวณค่า alpha
print(f'Alpha: {alpha}')

# แสดงภาพ
img_to_plot = img[0].detach().cpu().numpy()  # ย้าย Tensor ไป CPU และแปลงเป็น NumPy
img_to_plot = np.transpose(img_to_plot, (1, 2, 0))  # เปลี่ยนรูปแบบจาก [C, H, W] เป็น [H, W, C]

# ปรับค่าให้อยู่ในช่วง 0-1 (กรณี Tensor ของภาพถูก Normalize)
if img_to_plot.min() < 0 or img_to_plot.max() > 1:
    img_to_plot = (img_to_plot - img_to_plot.min()) / (img_to_plot.max() - img_to_plot.min())

plt.imshow(img_to_plot)  # แสดงภาพ
plt.title(f'Alpha: {alpha}')
plt.show()


In [ ]:
# ตรวจสอบขนาดของ output
print(f"Shape of output: {output.shape}")

# ตรวจสอบว่ามีจำนวน batch ใน output เท่าใด
batch_size = output.size(0)  # ขนาดในมิติแรกของ output

# เลือก index ที่เหมาะสม (เช่น index 0 ถ้า batch_size >= 1)
index = 0  # ปรับ index ตามความต้องการ

if index < batch_size:
    distance_left, distance_right = cal_distance(output[index, :].detach().cpu().numpy())
    print(f'Distance Left: {distance_left}')
    print(f'Distance Right: {distance_right}')
    alpha = cal_alpha(distance_left, distance_right)
    print(f'Alpha: {alpha}')
    plt.imshow(np.transpose(img[index].cpu().numpy(), (1, 2, 0)))  # ต้องแปลงกลับไปที่ CPU ด้วยถ้าเป็น Tensor บน GPU
else:
    print(f"Index {index} is out of bounds for dimension 0 with size {batch_size}")


In [ ]:
def plot_points_on_image_in_batches(data_loader, model, device):
    def classify_rotation(alpha):
        """Classify rotation based on alpha value."""
        if -1 <= alpha < -0.2:
            return "Right rotate"
        elif -0.2 <= alpha <= 0.2:
            return "Normal"
        elif 0.2 < alpha <= 1:
            return "Left rotate"
        else:
            return "Undefined"

    gt_labels = []   
    pred_labels = []  
    total_imgs = 0

    for batch_idx, (images, gt_points_raw, filenames) in enumerate(data_loader):
        images = images.to(device)
        gt_points_raw = gt_points_raw.to(device)

        with torch.no_grad():
            pred_points_raw = model(images)

        # Process each image ใน batch
        for i in range(len(images)):
            # เปลี่ยนรูปแบบ image เป็น H x W x C
            show_img = np.transpose(images[i].cpu().numpy(), (1, 2, 0))
            gt_points = gt_points_raw[i].cpu().numpy()
            pred_points = pred_points_raw[i].cpu().numpy()
            image_filename = filenames[i]

            # คำนวณสำหรับ Ground Truth
            distance_left_gt = round(gt_points[1] - gt_points[2])
            distance_right_gt = round(gt_points[2] - gt_points[0])
            alpha_gt = round((distance_right_gt - distance_left_gt) / (distance_right_gt + distance_left_gt + 1e-6), 3)
            rotation_gt = classify_rotation(alpha_gt)
            gt_labels.append(rotation_gt)

            # สำหรับ Pred: ปัดค่า X1, X2, X3 ให้เป็นจำนวนเต็มก่อนคำนวณ
            pred_points = np.rint(pred_points).astype(int)
            distance_left_pred = round(pred_points[1] - pred_points[2])
            distance_right_pred = round(pred_points[2] - pred_points[0])
            alpha_pred =round((distance_right_pred - distance_left_pred) / (distance_right_pred + distance_left_pred + 1e-6), 3)
            rotation_pred = classify_rotation(alpha_pred)
            pred_labels.append(rotation_pred)

            # แสดงข้อมูลใน Console
            print(f"Batch {batch_idx + 1}, Image {i + 1} ({image_filename}):")
            print(f"  GT -> X1: {gt_points[0]}, X2: {gt_points[1]}, X3: {gt_points[2]}, Alpha: {alpha_gt:.3f}, Rotation: {rotation_gt}")
            print(f"       Distance Left: {distance_left_gt}, Distance Right: {distance_right_gt}")
            print(f"  Pred -> X1: {pred_points[0]}, X2: {pred_points[1]}, X3: {pred_points[2]}, Alpha: {alpha_pred:.3f}, Rotation: {rotation_pred}")
            print(f"       Distance Left: {distance_left_pred}, Distance Right: {distance_right_pred}")

            # สำหรับการ plot ใช้สเกลของ y เท่ากับจำนวนจุดที่ต้องการ (3 จุด)
            y_values = np.linspace(0, show_img.shape[0], 3)

            plt.figure(figsize=(12, 6))

            # ---- Plot สำหรับ Ground Truth ----
            plt.subplot(1, 2, 1)
            # title แสดงชื่อ rotation และชื่อไฟล์ภาพ
            plt.title(f"GT - {rotation_gt} - {image_filename}")
            plt.imshow(show_img)
            plt.axis("off")
            # ใช้เส้นแนวนอนเพื่อแสดงระยะ (ตามโค้ดเก่า)
            # (เส้นสำหรับตำแหน่ง X1, X2, X3 ถูกวาดด้วย plt.plot หากต้องการให้แสดงก็เหลือไว้ แต่ถ้าต้องการเอาออกให้ลบออก)
            plt.plot([gt_points[0]] * len(y_values), y_values, "r-")
            plt.plot([gt_points[1]] * len(y_values), y_values, "r-")
            plt.plot([gt_points[2]] * len(y_values), y_values, "r-")
            plt.hlines(y=show_img.shape[0] // 4, xmin=gt_points[2], xmax=gt_points[1],
                       colors="blue", linestyles="dashed", label="Distance Left")
            plt.hlines(y=show_img.shape[0] // 4 + 10, xmin=gt_points[0], xmax=gt_points[2],
                       colors="green", linestyles="dashed", label="Distance Right")
            plt.legend()

            # ---- Plot สำหรับ Pred ----
            plt.subplot(1, 2, 2)
            plt.title(f"Pred - {rotation_pred} - {image_filename}")
            plt.imshow(show_img)
            plt.axis("off")
            plt.plot([pred_points[0]] * len(y_values), y_values, "r-")
            plt.plot([pred_points[1]] * len(y_values), y_values, "r-")
            plt.plot([pred_points[2]] * len(y_values), y_values, "r-")
            plt.hlines(y=show_img.shape[0] // 4, xmin=pred_points[2], xmax=pred_points[1],
                       colors="blue", linestyles="dashed", label="Distance Left")
            plt.hlines(y=show_img.shape[0] // 4 + 10, xmin=pred_points[0], xmax=pred_points[2],
                       colors="green", linestyles="dashed", label="Distance Right")
            plt.legend()

            plt.show()

        total_imgs += len(images)
        
    # Compute Confusion Matrix
    labels = ["Left rotate", "Normal", "Right rotate"]
    cm = confusion_matrix(gt_labels, pred_labels, labels=labels)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    display.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()

    # Calculate Metrics
    accuracy = accuracy_score(gt_labels, pred_labels)
    sensitivity = recall_score(gt_labels, pred_labels, average=None, labels=labels)
    specificity = []
    class_accuracies = []  # Accuracy per class
    for i in range(len(cm)):
        # Specificity
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity.append(tn / (tn + fp + 1e-6))

        # Accuracy per class
        class_correct = cm[i, i]
        class_total = cm[i, :].sum()
        class_accuracies.append(class_correct / class_total if class_total > 0 else 0)

    f1 = f1_score(gt_labels, pred_labels, average=None, labels=labels)

    # Compute Mean ± SD for metrics
    def calculate_mean_sd(values):
        return np.mean(values), np.std(values)

    mean_sensitivity, sd_sensitivity = calculate_mean_sd(sensitivity)
    mean_specificity, sd_specificity = calculate_mean_sd(specificity)
    mean_accuracy, sd_accuracy = calculate_mean_sd(class_accuracies)
    mean_f1, sd_f1 = calculate_mean_sd(f1)

    # Print Metrics
    print(f"\nOverall Metrics:")
    print(f"  Overall Accuracy: {accuracy:.3f}")
    for idx, label in enumerate(labels):
        print(f"  {label} -> Sensitivity: {sensitivity[idx]:.3f}, Specificity: {specificity[idx]:.3f}, Accuracy: {class_accuracies[idx]:.3f}, F1-score: {f1[idx]:.3f}")

    print(f"\nMean ± SD:")
    print(f"  Sensitivity: {mean_sensitivity:.3f} ± {sd_sensitivity:.3f}")
    print(f"  Specificity: {mean_specificity:.3f} ± {sd_specificity:.3f}")
    print(f"  Accuracy: {mean_accuracy:.3f} ± {sd_accuracy:.3f}")
    print(f"  F1-score: {mean_f1:.3f} ± {sd_f1:.3f}")

    print(f"\nTotal Images Processed: {total_imgs}")

# Usage
plot_points_on_image_in_batches(test_data_loader, model, device)



# Test on train

In [ ]:
img, output = next(iter(train_data_loader))
pred = model(img)
print(f'GT {output}')
print(f'Pred: {pred}')

In [ ]:
print(f'MAE: {MAE(output,pred)}')
print(f'MSE: {MSE(output,pred)}')